In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
import json, os, tempfile, shutil, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

RESULT_DIR = RESULTS
OUTPUT_DIR = RESULTS / 'extreme_ETS_radar'
LEADS = (1, 2, 3, 4, 5, 6)
LEVELS = np.array([850., 925., 1000.])
QUANTILES = (0.90, 0.95, 0.99)
W_TO_OMEGA = -10.0  # 原文件 w=-0.1*omega；恢复Pa/s一次；若已是omega，改为1
BLOCK_SIZE = 128
TIME_ROUND_TOL_NS = 1000
MIN_EXTREME_SAMPLES = 3  # R至少3个真实极端时次；不足或零方差为NaN
FIG_DPI = 800
SHOW_FIGURES = True
MAKE_ETS_FIGURE = True
FILE_OVERRIDES = {}
RAW_DIR = DATA
RAW_FILE_OVERRIDES = {}          # 可选：{('u',850): Path(r'实际文件.nc')}，默认文件名已按训练代码填写
RAW_START = '2000-01-01T06:00:00'
RAW_END = '2023-12-31T23:00:00'
RAW_LAT_RANGE = (24.0,36.0)
RAW_LON_RANGE = (108.0,126.0)
TIMESTEP = 6
TEST_SIZE = 0.2
VALID_SIZE = 0.1
TRUTH_CHECK_RTOL = 2e-5          # 原始场与反标准化testy的数值舍入容差
TRUTH_CHECK_ATOL = 2e-5
THRESHOLD_CSV = None            # 可选复用先前生成的训练阈值；默认重新从原ERA5训练段计算
BASIN_MASK_PATH = None  # 可选：同网格二维NC掩膜，1=流域内，0/NaN=流域外；不用时继承源NC有效域
BASIN_MASK_VARIABLE = 'mask'
QUANTILE_TEMP_DIR = str(WORK)
WORK.mkdir(parents=True, exist_ok=True)
MAX_QUANTILE_DISK_GB = 32.0  # 单层精确分位数临时空间上限；超限停止，不改成近似估计
RADAR_DIVISORS = None
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':11, 'axes.titlesize':12,
                     'axes.labelsize':11, 'xtick.labelsize':10.5, 'ytick.labelsize':10.5,
                     'pdf.fonttype':42, 'ps.fonttype':42})
if not np.isfinite(W_TO_OMEGA) or W_TO_OMEGA == 0: raise ValueError('W_TO_OMEGA必须为非零有限系数。')
if MIN_EXTREME_SAMPLES < 3: raise ValueError('MIN_EXTREME_SAMPLES至少为3。')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
ALIASES = {
    'time': ('time', 'valid_time', 'times', 'date', 'datetime'),
    'level': ('level', 'levels', 'pressure_level', 'isobaricInhPa', 'isobaricInPa', 'plev'),
    'latitude': ('latitude', 'lat'),
    'longitude': ('longitude', 'lon'),
}


def rounded_hour_times(values, source):
    raw = np.asarray(values)
    if not np.issubdtype(raw.dtype, np.datetime64):
        raise TypeError(f'{source}: time须可解码为datetime64，不能将非标准日历静默转为标准日历。')
    raw = raw.astype('datetime64[ns]')
    if np.isnat(raw).any() or raw.size == 0:
        raise ValueError(f'{source}: 空时间轴或NaT。')
    ns = raw.astype(np.int64)
    if np.any(np.diff(ns) <= 0):
        raise ValueError(f'{source}: 原时间轴重复或倒序。')
    hour = 3_600_000_000_000
    quotient, remainder = np.divmod(ns, hour)
    rounded_ns = (quotient + (remainder >= hour // 2)) * hour
    delta = rounded_ns - ns
    if np.max(np.abs(delta)) > TIME_ROUND_TOL_NS:
        raise ValueError(f'{source}: 非整点偏差{np.max(np.abs(delta))} ns超过允许的编码噪声。')
    if np.any(np.diff(rounded_ns) <= 0):
        raise ValueError(f'{source}: 整点编码修正后出现重复或倒序。')
    changes = [dict(index=int(i), original=str(raw[i]), rounded=str(np.datetime64(int(rounded_ns[i]), 'ns')),
                    correction_ns=int(delta[i])) for i in np.flatnonzero(delta)]
    return rounded_ns.astype('datetime64[ns]'), changes


def canonical_field(ds, variable, source):
    if variable in ds.data_vars:
        da = ds[variable]
    else:
        candidates = []
        for name, value in ds.data_vars.items():
            if not np.issubdtype(value.dtype, np.number):
                continue
            if all(any(alias in value.dims for alias in aliases) for aliases in ALIASES.values()):
                candidates.append(name)
        if len(candidates) != 1:
            raise ValueError(f'{source}: 找不到{variable}且无法唯一确定四维风场，候选={candidates}。')
        da = ds[candidates[0]]
    rename = {}
    for standard, aliases in ALIASES.items():
        found = [a for a in aliases if a in da.dims]
        if len(found) != 1:
            raise ValueError(f'{source}: {standard}维度不存在或有歧义：{found}。')
        rename[found[0]] = standard
    da = da.rename({a: b for a, b in rename.items() if a != b})
    for dim in tuple(da.dims):
        if dim not in ALIASES:
            if da.sizes[dim] != 1:
                raise ValueError(f'{source}: 不能自动丢弃额外非单例维度{dim}。')
            da = da.isel({dim: 0}, drop=True)
    for dim in ALIASES:
        if dim not in da.coords or da[dim].ndim != 1:
            raise ValueError(f'{source}: {dim}必须有一维坐标。')
    lev = np.asarray(da.level.values, dtype=np.float64)
    units = str(da.level.attrs.get('units', '')).strip().lower().replace(' ', '')
    if units in ('pa', 'pascal', 'pascals') or (not units and np.all(lev > 2000)):
        lev = lev / 100.0
    elif units not in ('', 'hpa', 'hectopascal', 'hectopascals', 'mb', 'millibar', 'millibars'):
        raise ValueError(f'{source}: 未识别的气压层单位{units!r}。')
    if len(np.unique(lev)) != len(lev):
        raise ValueError(f'{source}: 气压层坐标重复。')
    indices = []
    for wanted in LEVELS:
        hit = np.flatnonzero(np.isclose(lev, wanted, rtol=0, atol=1e-8))
        if len(hit) != 1:
            raise ValueError(f'{source}: 缺少或重复气压层{wanted} hPa。')
        indices.append(int(hit[0]))
    da = da.assign_coords(level=lev).isel(level=indices)
    for dim in ('latitude', 'longitude'):
        c = np.asarray(da[dim].values, dtype=np.float64)
        if not np.isfinite(c).all() or len(np.unique(c)) != len(c):
            raise ValueError(f'{source}: {dim}坐标含非有限值或重复。')
        da = da.isel({dim: np.argsort(c)}).assign_coords({dim: np.sort(c)})
    da = da.transpose('time', 'level', 'latitude', 'longitude')
    return da


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)


def file_stat(path):
    stat = Path(path).stat()
    return dict(path=str(Path(path).resolve()), size=stat.st_size, mtime_ns=stat.st_mtime_ns)


def merge_moments(old_n, old_mean_x, old_mean_y, old_m2_x, old_m2_y, old_cov,
                  x, y, valid):
    bn = valid.sum(axis=0).astype(np.int64)
    bx = np.divide(np.where(valid, x, 0.0).sum(axis=0), bn,
                   out=np.zeros(bn.shape, dtype=np.float64), where=bn > 0)
    by = np.divide(np.where(valid, y, 0.0).sum(axis=0), bn,
                   out=np.zeros(bn.shape, dtype=np.float64), where=bn > 0)
    dx = np.where(valid, x - bx, 0.0)
    dy = np.where(valid, y - by, 0.0)
    bm2x = np.sum(dx * dx, axis=0)
    bm2y = np.sum(dy * dy, axis=0)
    bcov = np.sum(dx * dy, axis=0)
    total = old_n + bn
    fraction = np.divide(bn, total, out=np.zeros(bn.shape, dtype=np.float64), where=total > 0)
    factor = np.divide(old_n.astype(np.float64) * bn, total,
                       out=np.zeros(bn.shape, dtype=np.float64), where=total > 0)
    shift_x = bx - old_mean_x
    shift_y = by - old_mean_y
    old_mean_x += shift_x * fraction
    old_mean_y += shift_y * fraction
    old_m2_x += bm2x + shift_x * shift_x * factor
    old_m2_y += bm2y + shift_y * shift_y * factor
    old_cov += bcov + shift_x * shift_y * factor
    return bn


def nc_path(lead, component, kind):
    return Path(FILE_OVERRIDES.get((int(lead), component, kind),
        RESULT_DIR / f'da_transunet_{kind}_{component}_{lead}_24year_changjiang_residual.nc'))

def scan_test():
    records, common, lat, lon = {}, None, None, None
    for h in LEADS:
        for c in ('u','v','w'):
            for kind in ('testy','predicty'):
                p = nc_path(h,c,kind)
                if not p.is_file(): raise FileNotFoundError(f'缺少最终订正风场：{p}')
                with xr.open_dataset(p, cache=False) as ds:
                    a=canonical_field(ds, f'{kind}_{c}', p)
                    tt,_=rounded_hour_times(a.time.values,p)
                    if lat is None: lat,lon=a.latitude.values,a.longitude.values
                    if not (np.array_equal(lat,a.latitude.values) and np.array_equal(lon,a.longitude.values)):
                        raise ValueError(f'网格不一致：{p}')
                    records[h,c,kind]={'path':p,'time':tt,'stat':file_stat(p)}
                    common=tt if common is None else np.intersect1d(common,tt)
    if common is None or len(common)<3: raise ValueError('测试共同时间不足。')
    for rec in records.values(): rec['indices']=np.searchsorted(rec['time'],common)
    return records,common,lat,lon

def selection(idx):
    return slice(int(idx[0]),int(idx[-1])+1) if len(idx) and (len(idx)==1 or np.all(np.diff(idx)==1)) else idx

def open_test_fields(stack,h,records):
    fields={}
    for c in ('u','v','w'):
        for kind in ('testy','predicty'):
            rec=records[h,c,kind]
            ds=stack.enter_context(xr.open_dataset(rec['path'],cache=False))
            fields[c,kind]=canonical_field(ds,f'{kind}_{c}',rec['path'])
    return fields

def read_speed(fields,records,h,start,stop,wind):
    values=[]
    for kind in ('testy','predicty'):
        cs=('u','v') if wind=='horizontal' else ('w',)
        arrays=[np.asarray(fields[c,kind].isel(time=selection(records[h,c,kind]['indices'][start:stop])).values,dtype=float) for c in cs]
        values.append(np.hypot(*arrays) if wind=='horizontal' else arrays[0]*W_TO_OMEGA)
    if basin_mask is not None:
        values=[np.where(basin_mask[None,None,:,:],a,np.nan) for a in values]
    return values

def load_basin_mask(latitude,longitude):
    if BASIN_MASK_PATH is None:
        warnings.warn('未指定流域掩膜：继承NC有效格点；只有源NC已按流域裁剪时才能称为流域统计。')
        return None
    with xr.open_dataset(BASIN_MASK_PATH) as ds:
        if BASIN_MASK_VARIABLE not in ds: raise KeyError('掩膜变量不存在。')
        a=ds[BASIN_MASK_VARIABLE]
        rename={}
        for standard in ('latitude','longitude'):
            hits=[x for x in ALIASES[standard] if x in a.dims]
            if len(hits)!=1: raise ValueError('掩膜必须带明确的经纬度维度。')
            rename[hits[0]]=standard
        a=a.rename({k:v for k,v in rename.items() if k!=v}).squeeze(drop=True)
        if set(a.dims)!={'latitude','longitude'}: raise ValueError('掩膜必须为二维。')
        a=a.sortby('latitude').sortby('longitude').transpose('latitude','longitude')
        if not (np.array_equal(a.latitude.values,latitude) and np.array_equal(a.longitude.values,longitude)):
            raise ValueError('掩膜与风场网格不同；不自动插值。')
        z=np.asarray(a.values)
        if not np.isin(z[np.isfinite(z)],[0,1]).all(): raise ValueError('掩膜只能取0/1/NaN。')
        mask=np.isfinite(z)&(z==1)
        if not mask.any(): raise ValueError('掩膜没有任何流域格点。')
        return mask

def event_mask(values,threshold,wind):
    return (np.abs(values) if wind=='vertical' else values)>=threshold

def validate_thresholds(frame,test_start):
    required={'wind','level_hpa','quantile','threshold','units','reference_split','reference_start','reference_end','source'}
    if not required.issubset(frame.columns): raise ValueError(f'阈值CSV缺少列：{required-set(frame.columns)}')
    for wind in ('horizontal','vertical'):
        for lev in LEVELS:
            vals=[]
            for q in QUANTILES:
                row=frame[(frame.wind==wind)&(frame.level_hpa==lev)&np.isclose(frame['quantile'],q)]
                if len(row)!=1: raise ValueError(f'阈值条目必须恰有一行：{wind},{lev},{q}')
                row=row.iloc[0]
                if row.reference_split!='train': raise ValueError('阈值必须来自训练集。')
                if row.units!=('m/s' if wind=='horizontal' else 'Pa/s'): raise ValueError('阈值单位不符。')
                if not np.isfinite(row.threshold) or row.threshold<0: raise ValueError('阈值必须有限且非负。')
                if pd.isna(row.source) or not str(row.source).strip(): raise ValueError('必须记录阈值来源。')
                begin=pd.Timestamp(row.reference_start); end=pd.Timestamp(row.reference_end)
                if begin.tzinfo is not None: begin=begin.tz_convert('UTC').tz_localize(None)
                if end.tzinfo is not None: end=end.tz_convert('UTC').tz_localize(None)
                if begin>end or end>=pd.Timestamp(test_start): raise ValueError('训练阈值时间与测试期重叠或无效。')
                vals.append(float(row.threshold))
            if np.any(np.diff(vals)<0): raise ValueError('阈值没有随分位数非递减。')
    return frame

def earliest_test_time(common_times):
    return min(rec['time'][0] for rec in records.values()) if 'records' in globals() and records else np.asarray(common_times)[0]


def raw_truth_path(component,level):
    prefix={'u':'U-component-of-wind','v':'V-component-of-wind','w':'Vertical-velocity'}[component]
    return Path(RAW_FILE_OVERRIDES.get((component,int(level)),RAW_DIR/f'{prefix}-{int(level)}hpa.nc'))


def canonical_raw_level(ds,component,level,path):
    if component not in ds: raise ValueError(f'{path}: 缺少原训练变量{component}。')
    a=ds[component]
    for dim in list(a.dims):
        if dim not in sum((ALIASES[k] for k in ('time','latitude','longitude')),()):
            if a.sizes[dim]!=1: raise ValueError(f'{path}: 单层源文件存在额外非单例维度{dim}。')
            if dim in ALIASES['level'] and dim in a.coords:
                v=float(a[dim].values[0]);units=str(a[dim].attrs.get('units','')).lower()
                if units=='pa' or v>2000: v/=100
                if not np.isclose(v,level): raise ValueError(f'{path}: 文件内气压层与文件名不一致。')
            a=a.isel({dim:0},drop=True)
    rename={}
    for name in ('time','latitude','longitude'):
        hits=[d for d in ALIASES[name] if d in a.dims]
        if len(hits)!=1 or hits[0] not in a.coords: raise ValueError(f'{path}: 缺少唯一{name}坐标。')
        rename[hits[0]]=name
    a=a.rename({k:v for k,v in rename.items() if k!=v})
    tt,_=rounded_hour_times(a.time.values,path)
    a=a.assign_coords(time=tt)
    for dim in ('latitude','longitude'):
        values=a[dim].values
        if not np.isfinite(values).all() or len(np.unique(values))!=len(values): raise ValueError(f'{path}: {dim}坐标无效。')
        a=a.sortby(dim)
    start=np.datetime64(RAW_START,'ns');end=np.datetime64(RAW_END,'ns')
    if start not in tt or end not in tt: raise ValueError(f'{path}: 原训练完整起止时间不齐，不能按缺失后的长度重新切分。')
    a=a.sel(time=slice(start,end),latitude=slice(RAW_LAT_RANGE[0],RAW_LAT_RANGE[1]),longitude=slice(RAW_LON_RANGE[0],RAW_LON_RANGE[1]))
    if min(a.sizes.values())==0: raise ValueError(f'{path}: 原训练区域选择后为空。')
    if np.any(np.diff(a.time.values)!=np.timedelta64(1,'h')): raise ValueError(f'{path}: 原训练时间范围内缺小时。')
    return a.transpose('time','latitude','longitude')


def reconstruct_original_split(times):
    vy_times=times[1:]
    target_times=vy_times[TIMESTEP-1:]
    index=int((1-TEST_SIZE)*len(vy_times))
    trainval_times=target_times[:index]
    test_times=target_times[index:]
    train_index=int((1-VALID_SIZE/(1-TEST_SIZE))*len(trainval_times))
    train_times=trainval_times[:train_index]
    valid_times=trainval_times[train_index:]
    if min(len(train_times),len(valid_times),len(test_times))<1: raise ValueError('原训练切分后某个子集为空。')
    info=dict(mode='original_DA_TransUNet_1h_exact',raw_time_count=len(times),vy_count=len(vy_times),
              windowed_target_count=len(target_times),trainval_index=index,train_index=train_index,
              training_time_count=len(train_times),validation_time_count=len(valid_times),test_time_count=len(test_times),
              training_start=str(train_times[0]),training_end=str(train_times[-1]),
              validation_start=str(valid_times[0]),validation_end=str(valid_times[-1]),
              test_start=str(test_times[0]),test_end=str(test_times[-1]),
              timestep=TIMESTEP,test_size=TEST_SIZE,valid_size=VALID_SIZE,
              split_formula='vy=raw[1:]; vys=vy[timestep-1:]; index=int((1-test_size)*len(vy)); train_index=int((1-valid_size/(1-test_size))*index)')
    return train_times,valid_times,test_times,info


def build_thresholds(common_times,latitude,longitude):
    cutoff=earliest_test_time(common_times)
    if THRESHOLD_CSV is not None:
        return validate_thresholds(pd.read_csv(THRESHOLD_CSV),cutoff)
    paths={(c,int(lev)):raw_truth_path(c,lev) for c in ('u','v','w') for lev in LEVELS}
    missing=[str(p) for p in paths.values() if not p.is_file()]
    if missing: raise FileNotFoundError('原训练使用的ERA5完整风场文件缺失，请检查RAW_DIR；不需要trainy文件：\n'+'\n'.join(missing))
    source_stats={f'{c}_{lev}':file_stat(p) for (c,lev),p in paths.items()}
    rows=[]
    with ExitStack() as stack:
        fields={}; times=None
        for c in ('u','v','w'):
            per_level=[]
            for lev in LEVELS:
                path=paths[c,int(lev)]
                ds=stack.enter_context(xr.open_dataset(path,cache=False))
                a=canonical_raw_level(ds,c,lev,path)
                if not (np.array_equal(a.latitude.values,latitude) and np.array_equal(a.longitude.values,longitude)):
                    raise ValueError(f'{path}: 原训练风场裁剪区域与测试网格不一致，不自动插值。')
                if times is None: times=a.time.values
                elif not np.array_equal(a.time.values,times): raise ValueError('九个原始风场的时间轴不一致。')
                per_level.append(a)
            fields[c]=per_level  # 保留各层独立懒读取对象，避免非dask concat加载整个全集
        train_times,valid_times,test_times,split_info=reconstruct_original_split(times)
        if train_times[-1]>=cutoff: raise ValueError('重建训练段与原始测试期重叠；请核对输入文件版本。')
        checks=[]
        for c in ('u','v','w'):
            actual=records[1,c,'testy']['time']
            where=np.searchsorted(test_times,actual)
            if len(actual)==0 or np.any(where>=len(test_times)) or not np.array_equal(test_times[where],actual):
                raise ValueError(f'{c}: 保存的1h testy含原训练切分之外的测试目标时次。')
            if actual[-1]!=test_times[-1] or not np.array_equal(actual,test_times[where[0]:]):
                raise ValueError(f'{c}: 保存的1h testy不是原测试期的完整序列或连续尾段；请核对时间标签。')
            checks.append(dict(component=c,exact_full_test_match=bool(np.array_equal(actual,test_times)),
                omitted_test_prefix=int(where[0]),stored_test_count=len(actual)))
        split_info.update(test_checks=checks,source_files=source_stats,source_period=[RAW_START,RAW_END],
                          lat_range=list(RAW_LAT_RANGE),lon_range=list(RAW_LON_RANGE),
                          raw_w_to_omega=1.0,saved_test_w_to_omega=W_TO_OMEGA,
                          baseline='first-stage 1h training subset shared by all ETS lead times',basin_mask_path=str(BASIN_MASK_PATH))
        sample_check=[]
        for c in ('u','v','w'):
            rec=records[1,c,'testy']; actual=rec['time']
            ids=np.unique(np.linspace(0,len(actual)-1,3,dtype=int)); target=actual[ids]
            with xr.open_dataset(rec['path'],cache=False) as ds:
                saved=canonical_field(ds,f'testy_{c}',rec['path']).isel(time=ids).values.astype(float)
            if c=='w': saved*=W_TO_OMEGA
            raw=np.stack([a.sel(time=target).values.astype(float) for a in fields[c]],axis=1)
            finite=np.isfinite(raw)&np.isfinite(saved)
            if not finite.any(): raise ValueError(f'{c}: 真值一致性抽查无共同有限值。')
            if not np.allclose(raw[finite],saved[finite],rtol=TRUTH_CHECK_RTOL,atol=TRUTH_CHECK_ATOL):
                raise ValueError(f'{c}: 原始风场与已保存testy抽查不一致，可能源数据版本、单位或时间标签不匹配。')
            sample_check.append(dict(component=c,checked_times=[str(t) for t in target],max_absolute_difference=float(np.max(np.abs(raw[finite]-saved[finite])))))
        split_info['raw_vs_saved_truth_checks']=sample_check
        pd.DataFrame({'time':train_times}).to_csv(OUTPUT_DIR/'threshold_training_target_times.csv',index=False,encoding='utf-8-sig')
        atomic_json(OUTPUT_DIR/'threshold_training_split.json',split_info)
        print(f'已按原训练代码恢复训练段：{train_times[0]} → {train_times[-1]}；{len(train_times):,}个时次。')
        fields={c:[a.sel(time=train_times) for a in levels] for c,levels in fields.items()}
        capacity=len(train_times)*len(latitude)*len(longitude); nbytes=capacity*8
        tempbase=Path(QUANTILE_TEMP_DIR or tempfile.gettempdir())
        if nbytes>MAX_QUANTILE_DISK_GB*1024**3 or shutil.disk_usage(tempbase).free<nbytes*1.1:
            raise MemoryError(f'单层精确分位数需约{nbytes/1024**3:.2f}GiB临时磁盘，请调整临时目录/上限。')
        for wind in ('horizontal','vertical'):
            for li,lev in enumerate(LEVELS):
                with tempfile.TemporaryDirectory(prefix='wind_quantile_',dir=QUANTILE_TEMP_DIR) as tmp:
                    mm=np.memmap(Path(tmp)/'values.dat',dtype='float64',mode='w+',shape=(capacity,))
                    used=0
                    for start in range(0,len(train_times),BLOCK_SIZE):
                        sl=slice(start,min(start+BLOCK_SIZE,len(train_times)))
                        if wind=='horizontal':
                            z=np.hypot(fields['u'][li].isel(time=sl).values.astype(float),fields['v'][li].isel(time=sl).values.astype(float))
                        else:
                            z=np.abs(fields['w'][li].isel(time=sl).values.astype(float))
                        z=z[np.isfinite(z)&(True if basin_mask is None else basin_mask[None,:,:])]
                        mm[used:used+len(z)]=z; used+=len(z)
                    if used<3: raise ValueError('训练有效样本不足。')
                    positions=np.asarray(QUANTILES)*(used-1)
                    lo=np.floor(positions).astype(int);hi=np.ceil(positions).astype(int)
                    view=mm[:used];view.partition(np.unique(np.r_[lo,hi]))
                    qvals=np.array(view[lo])+(positions-lo)*(np.array(view[hi])-np.array(view[lo]))
                    del view;mm.flush();del mm
                for q,t in zip(QUANTILES,qvals):
                    components=('u','v') if wind=='horizontal' else ('w',)
                    rows.append(dict(wind=wind,level_hpa=int(lev),quantile=q,threshold=float(t),
                        units='m/s' if wind=='horizontal' else 'Pa/s',reference_split='train',
                        reference_start=str(train_times[0]),reference_end=str(train_times[-1]),
                        source=';'.join(str(paths[c,int(lev)].resolve()) for c in components),
                        n_training_finite=used,training_time_count=len(train_times),
                        quantile_method='linear, exact disk partition',training_split_mode=split_info['mode']))
                print(f'训练阈值：{wind} {lev:g}hPa = {qvals}',flush=True)
    for (c,lev),path in paths.items():
        if file_stat(path)!=source_stats[f'{c}_{lev}']: raise RuntimeError('阈值计算期间原始风场发生变化，请重新运行。')
    return validate_thresholds(pd.DataFrame(rows),cutoff)


In [ ]:
def conditional_grid_metrics(truth,prediction,threshold,wind='horizontal'):
    valid=np.isfinite(truth)&np.isfinite(prediction)
    event=valid&event_mask(truth,threshold,wind)
    return valid,event

def compute_extremes(records,common_times,latitude,longitude,thresholds):
    rows=[]; maps=[]
    shape=(len(LEVELS),len(latitude),len(longitude))
    for h in LEADS:
        with ExitStack() as stack:
            fields=open_test_fields(stack,h,records)
            for wind in ('horizontal','vertical'):
                qs=np.array([[float(thresholds.loc[(thresholds.wind==wind)&(thresholds.level_hpa==lev)&np.isclose(thresholds['quantile'],q),'threshold'].iloc[0]) for lev in LEVELS] for q in QUANTILES])
                n=np.zeros((len(QUANTILES),)+shape,dtype=np.int64)
                mx,my,m2x,m2y,cov,sse,sae=[np.zeros_like(n,dtype=float) for _ in range(7)]
                counts=np.zeros((len(QUANTILES),len(LEVELS),4),dtype=np.int64)
                vmin=np.full(shape,np.inf); vmax=np.full(shape,-np.inf)
                for start in range(0,len(common_times),BLOCK_SIZE):
                    stop=min(start+BLOCK_SIZE,len(common_times))
                    truth,pred=read_speed(fields,records,h,start,stop,wind)
                    valid=np.isfinite(truth)&np.isfinite(pred)
                    vmin=np.minimum(vmin,np.min(np.where(valid,truth,np.inf),axis=0))
                    vmax=np.maximum(vmax,np.max(np.where(valid,truth,-np.inf),axis=0))
                    for qi,q in enumerate(QUANTILES):
                        cut=qs[qi][None,:,None,None]
                        obs=event_mask(truth,cut,wind); fcst=event_mask(pred,cut,wind)
                        for j,m in enumerate((obs&fcst,~obs&fcst,obs&~fcst,~obs&~fcst)):
                            counts[qi,:,j]+=(valid&m).sum(axis=(0,2,3))
                        extreme=valid&obs
                        bn=merge_moments(n[qi],mx[qi],my[qi],m2x[qi],m2y[qi],cov[qi],truth,pred,extreme)
                        e=np.where(extreme,pred-truth,0.)
                        sse[qi]+=np.sum(e*e,axis=0); sae[qi]+=np.sum(np.abs(e),axis=0); n[qi]+=bn
                vrange=vmax-vmin
                for qi,q in enumerate(QUANTILES):
                    enough=n[qi]>=MIN_EXTREME_SAMPLES
                    den=np.sqrt(m2x[qi])*np.sqrt(m2y[qi])
                    r=np.divide(cov[qi],den,out=np.full(shape,np.nan),where=enough&(den>0))
                    r=np.clip(r,-1,1)
                    rmse=np.sqrt(np.divide(sse[qi],n[qi],out=np.full(shape,np.nan),where=enough))
                    mae=np.divide(sae[qi],n[qi],out=np.full(shape,np.nan),where=enough)
                    rmspe=np.divide(rmse,vrange,out=np.full(shape,np.nan),where=enough&np.isfinite(vrange)&(vrange>0))
                    ds=xr.Dataset({k:(('level','latitude','longitude'),v) for k,v in {'R':r,'RMSE':rmse,'MAE':mae,'RMSPE':rmspe,'n_extreme':n[qi],'reference_range':vrange}.items()},coords=dict(level=LEVELS,latitude=latitude,longitude=longitude))
                    ds=ds.expand_dims(lead_hour=[h],wind=[wind],quantile=[q]); maps.append(ds)
                    for li,lev in enumerate(LEVELS):
                        H,F,M,C=map(int,counts[qi,li]); N=H+F+M+C
                        random=(H+F)*(H+M)/N if N else np.nan
                        denom=H+F+M-random
                        ets=(H-random)/denom if N and denom>0 else np.nan
                        row=dict(wind=wind,lead_hour=h,level_hpa=int(lev),quantile=q,threshold=qs[qi,li],
                            H=H,F=F,M=M,C=C,N=N,random_hits=random,ETS=ets,
                            n_observed_extremes=H+M,event_rule='speed >= training threshold')
                        for k,a in [('R',r),('RMSE',rmse),('MAE',mae),('RMSPE',rmspe)]:
                            good=np.isfinite(a[li]); row[k]=float(a[li][good].mean()) if good.any() else np.nan
                            row[f'n_grid_{k}']=int(good.sum())
                        rows.append(row)
                print(f'评价完成：{h}h {wind}',flush=True)
    return pd.DataFrame(rows),xr.combine_by_coords(maps,combine_attrs='drop_conflicts')

def save_netcdf(ds,path):
    with tempfile.TemporaryDirectory(prefix='extreme_metrics_', dir=WORK) as tmp:
        p=Path(tmp)/'metrics.nc'; ds.to_netcdf(p,engine='netcdf4',encoding={k:{'zlib':True,'complevel':4} for k in ds.data_vars})
        shutil.copyfile(p,path)

records,common_times,latitude,longitude=scan_test()
basin_mask=load_basin_mask(latitude,longitude)
thresholds=build_thresholds(common_times,latitude,longitude)
thresholds.to_csv(OUTPUT_DIR/'training_extreme_thresholds.csv',index=False,encoding='utf-8-sig')
summary,grid_metrics=compute_extremes(records,common_times,latitude,longitude,thresholds)
for rec in records.values():
    if file_stat(rec['path'])!=rec['stat']: raise RuntimeError('计算期间源文件已改变；不保存可能混合版本的结果。')
summary.to_csv(OUTPUT_DIR/'extreme_metrics_ETS_by_lead_level.csv',index=False,encoding='utf-8-sig')
grid_metrics.attrs.update(event_definition='truth/predicted speed greater than or equal to train-only level-specific threshold',
    vertical_definition='events: abs(omega); continuous metrics: signed omega, Pa/s; omega = saved_w * W_TO_OMEGA',w_to_omega=float(W_TO_OMEGA),
    continuous_metrics='gridwise temporal metrics on observed extremes; unweighted spatial means',
    rmspe='conditional RMSE / full paired-valid test reference temporal range; not extreme-subset range; no x100',
    ets='pooled all paired-valid test grid-time samples per lead and level, including correct negatives and false alarms')
save_netcdf(grid_metrics,OUTPUT_DIR/'extreme_grid_metrics.nc')


In [ ]:
radar_rows = []
for wind in ('horizontal', 'vertical'):
    fields = grid_metrics.sel(wind=wind)
    metric_values = {}
    for name in ('R', 'RMSE', 'MAE', 'RMSPE'):
        values = fields[name]
        common_support = np.isfinite(values).all(('lead_hour', 'quantile'))
        spatial_means = values.where(common_support).mean(('latitude', 'longitude'), skipna=True)
        metric_values[name] = spatial_means.mean(('level', 'lead_hour'), skipna=False)
    for q in QUANTILES:
        row = {'wind': wind, 'quantile': q}
        for name, values in metric_values.items():
            row[name] = float(values.sel(quantile=q))
        radar_rows.append(row)
pd.DataFrame(radar_rows).to_csv(OUTPUT_DIR / 'radar_unscaled_values.csv', index=False)
